# Orsini LP Directory Workflow Examples

This notebook shows practical ways to use the `orsini_lp` library on one trace, one directory of traces, or multiple directories of traces. The examples are set up so you can point at any folder on your machine that contains two-column Langmuir probe files and write a `results/` directory inside that folder. Auto-manifest generation now requires explicit probe metadata so the notebook also shows how to derive cylindrical probe radius and collection area from the real probe dimensions.


## What This Notebook Covers

- Build an auto-generated manifest for a directory of trace files.
- Inspect and optionally override inferred metadata before processing.
- Run the low-level single-trace workflow step by step.
- Run the high-level `process_trace_directory(...)` helper that writes a full `results/` tree.
- Loop over multiple directories and collect a combined summary table.


In [4]:
from pathlib import Path
import sys
import numpy as np

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / "orsini_lp").exists():
    for parent in [project_root, *project_root.parents]:
        if (parent / "orsini_lp").exists():
            project_root = parent
            break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from orsini_lp import (
    BayesianConfig,
    build_directory_manifest,
    compare_trace,
    cylindrical_probe_metadata,
    fit_bayesian,
    load_trace,
    process_trace_directory,
    run_legacy,
)
from orsini_lp.compare import write_trace_artifacts
from orsini_lp.plotting import plot_eedf_fit, plot_iv_fit

pd.set_option("display.max_columns", None)
project_root


PosixPath('/Users/braden/Documents/ProbeTools/Testbed')

## 1. Point The Notebook At A Trace Directory

Replace `trace_dir` below with any directory on your machine that contains Langmuir trace files. The helper functions will scan that directory for supported two-column `.txt` and `.csv` files, plus the local HDF5 example format. You must also provide explicit probe metadata for the directory.


In [5]:
# Replace this with any directory of Langmuir probe traces.
data_dir = Path('/Users/braden/Documents/Bismuth Cathode/2026-03 Zn-Kr Campaign/Orsini_test_directory')
#trace_dir = data_dir / 'Axial Langmuir'
trace_dir = data_dir / 'RPA Langmuir'
trace_dir = Path(trace_dir).resolve()
results_dir = trace_dir / "results"

# Supply the real probe dimensions used for every trace in this directory.
# Example here: 0.100 in exposed length, 0.002 in diameter cylindrical probe.
probe_metadata = cylindrical_probe_metadata(
    #exposed_length_m=0.100 * 0.0254,
    #diameter_m=0.002 * 0.0254,
    exposed_length_m=17/64 * 0.0254,
    diameter_m=0.005 * 0.0254
)

# Optional: set this when the filenames do not encode gas.
directory_gas = None
# directory_gas = "Kr"

probe_metadata


{'probe_geometry': 'cylindrical',
 'probe_radius_m': 6.35e-05,
 'probe_area_m2': 2.7045511696828938e-06}

## 2. Auto-Discover The Files And Inspect The Inferred Metadata

This is the safest first step on a new folder. It lets you confirm the filenames, gas, flow rate, discharge current, and the explicit probe geometry metadata that will be stamped onto every discovered trace before processing.


In [6]:
manifest = build_directory_manifest(trace_dir, gas=directory_gas, **probe_metadata)
#display(
#    manifest[
#        [
#            "trace_id",
#            "trace_path",
#            "gas",
#            "flow_sccm",
#            "discharge_current_a",
#            "probe_area_m2",
#            "notes",
#        ]
#    ]
#)


## 3. Optionally Override Any Inferred Metadata

Filename-based inference is convenient, but you should still correct anything that is experiment-specific. A common pattern is to adjust `gas`, `probe_area_m2`, `probe_radius_m`, or operating-point metadata before processing.


In [7]:
manifest = manifest.copy()

# Examples of manual overrides:
# manifest.loc[manifest["trace_id"].str.contains("zn", case=False), "gas"] = "Zn"
# If a different directory uses a different cylindrical probe, rebuild probe_metadata like this:
# probe_metadata = cylindrical_probe_metadata(exposed_length_m=0.233 * 0.0254, diameter_m=0.010 * 0.0254)
# manifest = build_directory_manifest(trace_dir, **probe_metadata)
# manifest.loc[manifest["trace_id"] == "my_trace", "probe_area_m2"] = 4.77e-06
# manifest.loc[manifest["trace_id"] == "my_trace", "probe_radius_m"] = 1.27e-04

display(manifest[["trace_id", "gas", "flow_sccm", "discharge_current_a", "probe_area_m2", "probe_radius_m"]])


,trace_id,gas,flow_sccm,discharge_current_a,probe_area_m2,probe_radius_m
0,kr_15sccm_10a_lp,Kr,15.0,10.0,0.000003,0.000063
1,kr_15sccm_10a_lp2,Kr,15.0,10.0,0.000003,0.000063
2,kr_15sccm_11a_lp,Kr,15.0,11.0,0.000003,0.000063
3,kr_15sccm_12a_lp,Kr,15.0,12.0,0.000003,0.000063
4,kr_15sccm_12p5a_lp,Kr,15.0,12.5,0.000003,0.000063
5,kr_15sccm_13a_lp,Kr,15.0,13.0,0.000003,0.000063
6,kr_15sccm_14a_lp,Kr,15.0,14.0,0.000003,0.000063
7,kr_15sccm_15a_lp,Kr,15.0,15.0,0.000003,0.000063
8,kr_15sccm_16a_lp,Kr,15.0,16.0,0.000003,0.000063
9,kr_15sccm_17a_lp,Kr,15.0,17.0,0.000003,0.000063


## 4. Low-Level Single-Trace Workflow

This section shows the individual library calls for one trace. It is useful when you want full control over the intermediate objects or want to experiment with priors, diagnostic plots, or custom post-processing.


In [8]:
trace_id = manifest.iloc[0]["trace_id"]
trace_row = manifest.loc[manifest["trace_id"] == trace_id].iloc[0]
trace = load_trace(trace_row)

config = BayesianConfig(
    nlive=40,
    dlogz=2.0,
    max_points=80,
    posterior_draws=40,
    random_seed=9,
)

bayes = fit_bayesian(trace, config=config)
legacy = run_legacy(trace)
comparison = compare_trace(trace, bayes, legacy)

comparison.comparison_table


,bayes_median,bayes_q16,bayes_q84,legacy_value,legacy_std,delta
metric,,,,,,
Vp,2.066738e+01,1.987980e+01,2.149673e+01,1.634662e+01,3.361637e-01,4.320762e+00
Te,6.737633e+00,6.436970e+00,7.135955e+00,1.797514e+00,4.431029e-02,4.940119e+00
ne,6.052783e+16,5.565871e+16,6.618290e+16,5.946822e+16,6.516208e+15,1.059605e+15
n_qn,6.052783e+16,5.565871e+16,6.618290e+16,3.924926e+16,4.205038e+16,2.127856e+16
p,2.848682e+00,2.649307e+00,2.948936e+00,NaN,NaN,NaN


In [9]:
display(bayes.summary.loc[["Vp", "Te", "ne", "p", "n_qn"]])

iv_fig = plot_iv_fit(trace, bayes, legacy)
eedf_fig = plot_eedf_fit(bayes)
iv_fig, eedf_fig


,median,mean,std,q16,q84,q2.25,q97.75
parameter,,,,,,,
Vp,2.066738e+01,2.073965e+01,7.964303e-01,1.987980e+01,2.149673e+01,1.938286e+01,2.222336e+01
Te,6.737633e+00,6.780189e+00,3.405984e-01,6.436970e+00,7.135955e+00,6.202446e+00,7.322951e+00
ne,6.052783e+16,6.125572e+16,5.361246e+15,5.565871e+16,6.618290e+16,5.261828e+16,7.214580e+16
p,2.848682e+00,2.809565e+00,1.423458e-01,2.649307e+00,2.948936e+00,2.558537e+00,2.997458e+00
n_qn,6.052783e+16,6.125572e+16,5.361246e+15,5.565871e+16,6.618290e+16,5.261828e+16,7.214580e+16


(<Figure size 700x450 with 1 Axes>, <Figure size 700x450 with 1 Axes>)

## 5. Write One Trace's Artifact Bundle To 
```<trace_dir>/results/<trace_id>/```

This mirrors the same file structure used by the batch tools: summary CSVs, posterior samples, plots, and metadata for the selected trace.


In [10]:
single_trace_results_dir = results_dir / trace.trace_id
write_trace_artifacts(single_trace_results_dir, trace, bayes, legacy, comparison)
sorted(path.name for path in single_trace_results_dir.iterdir())


['bayesian_summary.csv',
 'comparison_summary.csv',
 'eedf_fit.png',
 'eedf_posterior_predictive.csv',
 'iv_fit.png',
 'iv_posterior_predictive.csv',
 'legacy_derived_quantities.csv',
 'legacy_summary.csv',
 'metadata.json',
 'posterior_samples.csv',
 'trace_summary.csv']

## 6. Process An Entire Directory With One Function Call

This is the high-level workflow for day-to-day use. It automatically creates `<trace_dir>/results/`, writes an `auto_manifest.csv` copy of the discovered files, writes a full subdirectory for each trace, and writes a `batch_summary.csv` for the whole folder.


In [9]:
config = BayesianConfig(
    nlive=40,
    dlogz=2.0,
    max_points=80,
    posterior_draws=40,
    random_seed=9,
)

summary = process_trace_directory(
    trace_dir,
    gas=directory_gas,
    **probe_metadata,
    config=config,
    results_dir_name="results",
    # trace_ids=["lp_15sccm_10a1", "lp_15sccm_15a1"],  # optional subset
)

display(summary)
display(sorted(path.name for path in results_dir.iterdir()))


,trace_id,legacy_success,legacy_error,Vp_bayes_median,Vp_legacy_value,Vp_delta,Te_bayes_median,Te_legacy_value,Te_delta,ne_bayes_median,ne_legacy_value,ne_delta,n_qn_bayes_median,n_qn_legacy_value,n_qn_delta,p_bayes_median,p_legacy_value,p_delta
0,kr_15sccm_10a_lp,True,None,31.418007,18.943816,12.474191,6.644076,1.813757,4.830319,9.615809e+15,2.348759e+15,7.267050e+15,9.615809e+15,1.425448e+15,8.190361e+15,1.034578,NaN,NaN
1,kr_15sccm_10a_lp2,True,None,68.790495,17.349576,51.440919,9.410034,1.567294,7.842740,1.434115e+16,2.185788e+15,1.215536e+16,1.434115e+16,1.318084e+15,1.302307e+16,1.099363,NaN,NaN
2,kr_15sccm_11a_lp,True,None,25.126271,19.089382,6.036889,4.800135,1.751778,3.048357,7.566596e+15,3.115988e+15,4.450607e+15,7.566596e+15,1.892342e+15,5.674254e+15,1.304627,NaN,NaN
3,kr_15sccm_12a_lp,True,None,23.631042,18.288836,5.342206,4.661132,1.644534,3.016598,7.984406e+15,3.086915e+15,4.897491e+15,7.984406e+15,1.886490e+15,6.097916e+15,1.111368,NaN,NaN
4,kr_15sccm_12p5a_lp,True,None,29.023923,19.495041,9.528882,5.293990,1.806337,3.487653,1.240603e+16,3.099358e+15,9.306669e+15,1.240603e+16,1.867565e+15,1.053846e+16,1.017054,NaN,NaN
5,kr_15sccm_13a_lp,True,None,23.253004,17.080758,6.172246,4.507540,1.508850,2.998690,8.498802e+15,2.772920e+15,5.725882e+15,8.498802e+15,1.704055e+15,6.794746e+15,1.218781,NaN,NaN
6,kr_15sccm_14a_lp,True,None,40.868592,16.276185,24.592407,6.302547,1.419227,4.883320,2.773564e+16,2.600315e+15,2.513532e+16,2.773564e+16,1.588940e+15,2.614670e+16,1.023979,NaN,NaN
7,kr_15sccm_15a_lp,True,None,20.081526,17.618266,2.463260,5.656535,1.522059,4.134477,2.126996e+15,2.761502e+15,-6.345064e+14,2.126996e+15,1.675244e+15,4.517521e+14,2.921344,NaN,NaN
8,kr_15sccm_16a_lp,True,None,53.349912,15.910373,37.439540,8.752280,1.291139,7.461141,1.038054e+16,2.015462e+15,8.365081e+15,1.038054e+16,1.210940e+15,9.169603e+15,1.111618,NaN,NaN
9,kr_15sccm_17a_lp,True,None,21.002666,17.081478,3.921188,5.955085,1.452606,4.502480,2.014973e+15,2.265977e+15,-2.510033e+14,2.014973e+15,1.311576e+15,7.033970e+14,2.953156,NaN,NaN


['.DS_Store',
 'auto_manifest.csv',
 'batch_summary.csv',
 'kr_15sccm_10a_lp',
 'kr_15sccm_10a_lp2',
 'kr_15sccm_11a_lp',
 'kr_15sccm_12a_lp',
 'kr_15sccm_12p5a_lp',
 'kr_15sccm_13a_lp',
 'kr_15sccm_14a_lp',
 'kr_15sccm_15a_lp',
 'kr_15sccm_16a_lp',
 'kr_15sccm_17a_lp',
 'kr_15sccm_17p5a_lp',
 'kr_15sccm_18a_lp',
 'kr_15sccm_19a_lp',
 'kr_15sccm_20a_lp',
 'kr_15sccm_21a_lp',
 'kr_15sccm_22a_lp',
 'kr_15sccm_22p5a_lp',
 'kr_15sccm_23a_lp',
 'kr_15sccm_24a_lp',
 'kr_15sccm_25a_lp',
 'kr_15sccm_27p5a_lp',
 'kr_15sccm_27p5a_lp2',
 'kr_15sccm_30a_lp',
 'zn_15sccm_15a_lp',
 'zn_15sccm_15a_lp2',
 'zn_15sccm_15a_lp3']

## 7. Process Multiple Directories

If your data live in separate experiment folders, just loop over them. Each folder gets its own local `results/` directory, and you can still build a combined overview table in the notebook. If gas is not encoded in the filenames, include it in each job as well.


In [ ]:
directory_jobs = [
    {"trace_dir": trace_dir, "probe_metadata": probe_metadata, "gas": directory_gas},
    # {
    #     "trace_dir": Path("/Users/braden/Documents/AnotherExperiment/2026-04-02"),
    #     "probe_metadata": cylindrical_probe_metadata(exposed_length_m=0.233 * 0.0254, diameter_m=0.010 * 0.0254),
    # },
]

all_summaries = []
for job in directory_jobs:
    one_dir = Path(job["trace_dir"]).resolve()
    one_probe_metadata = job["probe_metadata"]
    one_gas = job.get("gas")
    one_summary = process_trace_directory(
        one_dir,
        gas=one_gas,
        **one_probe_metadata,
        config=config,
        results_dir_name="results",
    )
    one_summary.insert(0, "trace_directory", str(one_dir))
    all_summaries.append(one_summary)

batch_overview = pd.concat(all_summaries, ignore_index=True)
display(batch_overview)


## 8. Notes And CLI Equivalents

- `cylindrical_probe_metadata(...)` is the easiest way to derive `probe_radius_m` and `probe_area_m2` from your real cylindrical probe dimensions.
- `build_directory_manifest(...)` now requires explicit probe metadata and is the right place to inspect or override inferred metadata before a run.
- `process_trace_directory(...)` is the easiest high-level entry point for routine use once that probe metadata is defined.
- The folder helper writes both `auto_manifest.csv` and `batch_summary.csv` into `<trace_dir>/results/`.
- If you prefer the command line, the closest equivalents are:

```bash
python -m orsini_lp.cli compare sample_data/local_manifest.csv \
  --trace-id lp_15sccm_10a1 \
  --output-dir /tmp/example_compare

python -m orsini_lp.semilog LP_15sccm_10A1.txt \
  --output-dir /tmp/semilog_check
```
